# SR-PDS simulation -- quick laptop run

Sanity-checks the full pipeline before committing HPC time. Runs every
DGP and every estimator at a small replication count, builds every
output figure, and writes everything to `../results_quick/`.

Expected runtime on a recent MacBook: roughly 45-90 minutes. SR-PDS is
the bottleneck (~20s per fit, 2 fits per replication for vanilla
SR-PDS, 10 fits per replication for SR-PDS-CF).

Run this end-to-end first. If anything crashes or any plot looks wrong,
fix it here rather than queueing the full HPC job.


## 1. Setup

In [ ]:
import os
import sys
import pickle
import warnings
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '.')

from dgp import DGP_REGISTRY
from estimators import ESTIMATOR_REGISTRY, SR_PDS_LOG
from simulation import run_all, run_simulation
from evaluate import (evaluate_all, summary_table, print_summary,
                      evaluate_recovery)
from plots import (plot_bias, plot_coverage, plot_rmse_heatmap,
                   plot_summary_panel, plot_distributions,
                   plot_recovery_heatmap, plot_coefficient_recovery,
                   plot_n_trajectory, plot_recovery_vs_n)

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 90

N           = 500
P           = 50
S           = 6
BETA0       = 0.5
N_REPS      = 10        # small for laptop -- this is the whole point of the quick notebook
N_REPS_CF   = 5         # cross-fit is 5x more expensive per rep
N_JOBS      = 4         # outer parallelism, ignored by the SR-PDS estimators
RESULTS_DIR = '../results_quick'

os.makedirs(RESULTS_DIR, exist_ok=True)
DGP_KEYS = list(DGP_REGISTRY.keys())
EST_KEYS = list(ESTIMATOR_REGISTRY.keys())

print(f"DGPs ({len(DGP_KEYS)}):       {DGP_KEYS}")
print(f"Estimators ({len(EST_KEYS)}): {EST_KEYS}")
print(f"N={N}, p={P}, n_reps={N_REPS} (sr_pds_cf at {N_REPS_CF})")

---
## 2. DGP exploration

Quick check that all eight DGPs generate sensible data. The summary
shows the standard deviation of y, the standard deviation of d, and
whether the propensity has a non-trivial structure.

In [ ]:
print(f"{'DGP':<6} {'label':<26} {'E[y]':>8} {'std[y]':>8} {'std[d]':>8} {'conf':<6}")
print('-' * 70)
for key, entry in DGP_REGISTRY.items():
    X, d, y, _ = entry['fn'](n=2000, p=P, s=S, beta0=BETA0)
    has_conf = 'yes' if entry['truth_terms_d'] else ('linear' if d.std() > 1.2 else 'no')
    print(f"{key:<6} {entry['label']:<26} {y.mean():>8.2f} {y.std():>8.2f} "
          f"{d.std():>8.2f} {has_conf:<6}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, key in zip(axes.flat, ['dgp1', 'dgp6', 'dgp7', 'dgp8']):
    X, d, y, _ = DGP_REGISTRY[key]['fn'](n=500, p=P, s=S, beta0=BETA0)
    ax.scatter(d, y, alpha=0.4, s=12)
    ax.set_xlabel('d')
    ax.set_ylabel('y')
    ax.set_title(f"{key}: {DGP_REGISTRY[key]['label']}")
    ax.grid(alpha=0.3)
plt.suptitle('Outcome vs treatment across DGPs (n=500)', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Smoke test

Verify each estimator returns a sensible result before launching the
full Monte Carlo. The standard methods are tested across all DGPs in
one go; the SR-PDS methods get a single run on DGP-6 because they're
expensive.

In [ ]:
from estimators import full_ols, pds_lasso, dml_lasso, dml_rf, dml_nn

print(f"True beta0 = {BETA0}\n")
print(f"{'DGP':<6} {'Full':>8} {'PDS-L':>8} {'DML-L':>8} {'DML-RF':>8} {'DML-NN':>8}")
print('-' * 56)
for key, entry in DGP_REGISTRY.items():
    X, d, y, _ = entry['fn'](n=N, p=P, s=S, beta0=BETA0, seed=42)
    row = [key]
    for est in (full_ols, pds_lasso, dml_lasso, dml_rf, dml_nn):
        try:
            r = est(X, d, y, N, P)
            row.append(f"{r['beta_hat']:>+.3f}")
        except Exception:
            row.append('FAIL')
    print(f"{row[0]:<6} {row[1]:>8} {row[2]:>8} {row[3]:>8} {row[4]:>8} {row[5]:>8}")

In [ ]:
# single SR-PDS run on DGP-6. Should discover x4*x5 in d (the confounder)
# and the mixed-nonlinear terms in y.
from estimators import sr_pds

X, d, y, _ = DGP_REGISTRY['dgp6']['fn'](n=300, p=P, s=S, beta0=BETA0, seed=42)
result = sr_pds(X, d, y, 300, P, log=False, debug=False)

print(f"DGP-6 single SR-PDS run (n=300):")
print(f"  beta_hat = {result['beta_hat']:.4f}  (true = {BETA0})")
print(f"  CI       = [{result['ci_low']:.3f}, {result['ci_high']:.3f}]")
print(f"  pre-LASSO  y: {result['pre_lasso_terms_y']}")
print(f"  pre-LASSO  d: {result['pre_lasso_terms_d']}")
print(f"  post-LASSO y: {result['post_lasso_terms_y']}")
print(f"  post-LASSO d: {result['post_lasso_terms_d']}")

---
## 4. Headline Monte Carlo

All 8 DGPs x all 7 estimators at n=500. We use `reps_overrides` to
cap sr_pds_cf at half the rep count of everything else since cross-fit
is 5x more expensive per rep.

In [ ]:
SR_PDS_LOG.clear()

combined = run_all(
    dgp_registry       = DGP_REGISTRY,
    estimator_registry = ESTIMATOR_REGISTRY,
    dgp_keys           = DGP_KEYS,
    estimator_keys     = EST_KEYS,
    n                  = N,
    p                  = P,
    s                  = S,
    beta0              = BETA0,
    n_reps             = N_REPS,
    n_jobs             = N_JOBS,
    save_dir           = RESULTS_DIR,
    reps_overrides     = {'sr_pds_cf': N_REPS_CF},
)

combined.to_pickle(f'{RESULTS_DIR}/headline.pkl')
with open(f'{RESULTS_DIR}/sr_pds_log.pkl', 'wb') as f:
    pickle.dump(list(SR_PDS_LOG), f)
print(f"\nSaved headline + log to {RESULTS_DIR}")

---
## 5. Bias / RMSE / coverage / size

These are the four canonical metrics. Coverage and size are reported
as percentages; bias and RMSE are on the same scale as beta0.

In [ ]:
all_metrics = evaluate_all(combined, beta0=BETA0)
print_summary(all_metrics, BETA0)

In [ ]:
plot_bias(all_metrics, save=True);          plt.show()
plot_coverage(all_metrics, save=True);      plt.show()
plot_rmse_heatmap(all_metrics, save=True);  plt.show()
plot_summary_panel(all_metrics, save=True); plt.show()

In [ ]:
# per-DGP histograms -- look for bimodality or heavy tails that the
# point estimates would otherwise hide
for dgp_key in DGP_KEYS:
    fig = plot_distributions(combined, BETA0, dgp_key, save=True)
    plt.show()

---
## 6. Recovery analysis

How often did PySR actually discover the true nonlinear terms, and
how often did those discoveries survive the downstream LASSO step?

This only makes sense on DGPs that have non-trivial nonlinear truth.

In [ ]:
recovery_df = evaluate_recovery(list(SR_PDS_LOG), DGP_REGISTRY)
print(f"Recovery records: {len(recovery_df)}")
recovery_df.head(20)

In [ ]:
if not recovery_df.empty:
    plot_recovery_heatmap(recovery_df, equation='y', save=True); plt.show()
    sub_d = recovery_df[recovery_df['equation'] == 'd']
    if not sub_d.empty:
        plot_recovery_heatmap(recovery_df, equation='d', save=True); plt.show()
    plot_coefficient_recovery(recovery_df, equation='y', save=True); plt.show()

---
## 7. Abbreviated trajectory

A short trajectory at n in {30, 100, 500, 2000} to verify the trajectory
pipeline works end-to-end. The full trajectory (n=30 through n=50,000)
lives in `main_full.ipynb`. SR-PDS-CF is excluded because we only run
it at the headline n=500.

In [ ]:
N_GRID        = [30, 100, 500, 2000]
N_REPS_TRAJ   = 5
TRAJ_EST_KEYS = [k for k in EST_KEYS if k != 'sr_pds_cf']

SR_PDS_LOG.clear()
traj_rows = []
total = len(N_GRID) * len(DGP_KEYS) * len(TRAJ_EST_KEYS)
i = 0
for n_grid in N_GRID:
    for dgp_key in DGP_KEYS:
        for est_key in TRAJ_EST_KEYS:
            i += 1
            est = ESTIMATOR_REGISTRY[est_key]
            print(f"[{i}/{total}] n={n_grid:>5} {dgp_key} {est_key}", end='  ')
            df = run_simulation(
                DGP_REGISTRY[dgp_key]['fn'], est['fn'],
                estimator_key=est_key,
                n=n_grid, p=P, s=S, beta0=BETA0,
                n_reps=N_REPS_TRAJ,
                n_jobs=1 if est['requires_serial'] else N_JOBS,
                dgp_key=dgp_key,
            )
            df['dgp'] = dgp_key
            df['estimator'] = est_key
            df['dgp_label'] = DGP_REGISTRY[dgp_key]['label']
            df['est_label'] = est['label']
            df['beta0'] = BETA0
            df['n_grid'] = n_grid
            traj_rows.append(df)
            print(f"  beta={df['beta_hat'].mean():+.3f}")

traj_df = pd.concat(traj_rows, ignore_index=True)
traj_df.to_pickle(f'{RESULTS_DIR}/trajectory.pkl')
print(f"\nTrajectory saved to {RESULTS_DIR}/trajectory.pkl")

In [ ]:
plot_n_trajectory(traj_df, BETA0, save=True);          plt.show()
plot_recovery_vs_n(traj_df, DGP_REGISTRY, save=True); plt.show()

---
## 8. Output summary

In [ ]:
tag = datetime.datetime.now().strftime('%Y%m%d_%H%M')
print(f"Run tag: {tag}\n")
print(f"Files in {RESULTS_DIR}:")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {f}")